In [1]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib scipy

In [2]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 14})

# =====================================================================
# 1. INITIALIZATION & DIRECTORY SETUP
# =====================================================================
OUTPUT_DIR = "artifacts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def run_complete_model_2_pipeline(df, target_k=None):
    print("🚀 [START] Memulai Complete Pipeline Model 2 (Clustering Persona)...")

    # Konversi datetime dan ekstraksi periode bulan
    df['transaction_datetime'] = pd.to_datetime(df['transaction_datetime'])
    df['month'] = df['transaction_datetime'].dt.to_period('M').astype(str)

    # =====================================================================
    # 2. FEATURE AGGREGATION & ENGINEERING (Level User-Bulan)
    # =====================================================================
    print("📊 Mengagregasi data transaksi ke level bulanan per pengguna...")

    monthly_base = df.groupby(['user_id', 'month']).agg({
        'is_late_night': 'mean',
        'is_weekend': 'mean',
        'is_unbudgeted': 'mean',
        'is_risky_category': 'mean',
        'is_binge_spending': 'mean',
        'hourly_txn_count': 'mean',
        'amount': 'count',
        'income_monthly': 'first',
        'total_expense': 'first',
        'wants_spending': 'first',
        'total_debt_payment': 'first',
        'investment_amount': 'first'
    }).rename(columns={'amount': 'transaction_count'})

    # Rekayasa fitur rasio finansial (Raw Ratios)
    eps = 1e-5
    monthly_base['saving_rate_raw'] = (monthly_base['income_monthly'] - monthly_base['total_expense']) / (monthly_base['income_monthly'] + eps)
    monthly_base['wants_ratio_raw'] = monthly_base['wants_spending'] / (monthly_base['income_monthly'] + eps)
    monthly_base['investment_rate_raw'] = monthly_base['investment_amount'] / (monthly_base['income_monthly'] + eps)
    monthly_base['dti_ratio_raw'] = monthly_base['total_debt_payment'] / (monthly_base['income_monthly'] + eps)

    print("🍕 Mentransformasikan proporsi pengeluaran berdasarkan kategori...")
    cat_pivot = df.groupby(['user_id', 'month', 'category'])['amount'].sum().unstack(fill_value=0)
    cat_prop = cat_pivot.div(cat_pivot.sum(axis=1), axis=0)
    cat_prop.columns = [f'cat_{c.lower().replace(" ", "_")}' for c in cat_prop.columns]

    # Menggabungkan seluruh fitur ke dataset final training
    monthly = monthly_base.merge(cat_prop, on=['user_id', 'month'], how='left').fillna(0)

    # Seleksi kolom fitur untuk clustering
    features_to_cluster = [
        'is_late_night', 'is_weekend', 'is_unbudgeted', 'is_risky_category',
        'is_binge_spending', 'hourly_txn_count', 'transaction_count',
        'saving_rate_raw', 'wants_ratio_raw', 'investment_rate_raw', 'dti_ratio_raw'
    ] + list(cat_prop.columns)

    X = monthly[features_to_cluster].values

    # Standarisasi Data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    joblib.dump(scaler, os.path.join(OUTPUT_DIR, "cluster_scaler.pkl"))

    # =====================================================================
    # 3. CLUSTER EVALUATION (HYPERPARAMETER TUNING)
    # =====================================================================
    print("🔍 Mengevaluasi nilai K terbaik (k=2 hingga k=6)...")
    k_range = range(2, 7)
    inertias = []
    silhouettes = []
    db_indices = []

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_scaled)

        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X_scaled, labels))
        db_indices.append(davies_bouldin_score(X_scaled, labels))

    # Menentukan K optimal secara otomatis menggunakan Silhouette Score tertinggi jika tidak didefinisikan manual
    if target_k is None:
        optimal_k = k_range[np.argmax(silhouettes)]
        print(f"💡 K-optimal otomatis terpilih berdasarkan Silhouette Score: k={optimal_k}")
    else:
        optimal_k = target_k
        print(f"💡 Menggunakan nilai K yang ditentukan manual: k={optimal_k}")

    # Visualisasi Metrik Evaluasi (Elbow & Silhouette)
    fig, ax1 = plt.subplots(1, 2, figsize=(14, 5))

    ax1[0].plot(k_range, inertias, marker='o', color='b', linestyle='--')
    ax1[0].set_title('Metode Elbow (Inertia)')
    ax1[0].set_xlabel('Jumlah Cluster (k)')
    ax1[0].set_ylabel('Inertia')

    ax2 = ax1[1].twinx()
    ax1[1].plot(k_range, silhouettes, marker='s', color='g', label='Silhouette Score')
    ax2.plot(k_range, db_indices, marker='^', color='r', label='Davies-Bouldin Index')
    ax1[1].set_title('Silhouette Score vs Davies-Bouldin Index')
    ax1[1].set_xlabel('Jumlah Cluster (k)')
    ax1[1].set_ylabel('Silhouette Score', color='g')
    ax2.set_ylabel('Davies-Bouldin Index', color='r')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "cluster_evaluation_metrics.png"), dpi=150)
    plt.close()

    # =====================================================================
    # 4. FINAL CLUSTERING EXECUTION
    # =====================================================================
    print(f"🤖 Menjalankan model final K-Means dengan k={optimal_k}...")
    final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    monthly['cluster'] = final_kmeans.fit_predict(X_scaled)

    joblib.dump(final_kmeans, os.path.join(OUTPUT_DIR, "cluster_model.pkl"))

    # =====================================================================
    # 5. AUTOMATED PROFILING & PERSONA MAPPING
    # =====================================================================
    print("🏷️ Melakukan profiling centroid untuk pemetaan persona keuangan otomatis...")
    centroids = pd.DataFrame(scaler.inverse_transform(final_kmeans.cluster_centers_), columns=features_to_cluster)

    cluster_mapping = {}
    for idx, row in centroids.iterrows():
        # 1. SI MINIMALIS: Memiliki jumlah transaksi paling sedikit/minimum
        if row['transaction_count'] == centroids['transaction_count'].min():
            label = "Si Minimalis"
            desc = "Pengguna sangat jarang melakukan transaksi bulanan dan cenderung pasif, namun memiliki kontrol pengeluaran yang baik dengan tingkat tabungan yang sehat."

        # 2. SI HEMAT: Memiliki rasio tabungan paling tinggi/maksimum
        elif row['saving_rate_raw'] == centroids['saving_rate_raw'].max():
            label = "Si Hemat"
            desc = "Pengguna bertipe terencana, sangat disiplin dalam menyisihkan tabungan, serta ketat dalam menjaga rasio pengeluaran non-esensial (wants)."

        # 3. SI IMPULSIF: Kelompok sisanya (memiliki wants_ratio dan unbudgeted paling tinggi)
        else:
            label = "Si Impulsif"
            desc = "Pengguna rentan melakukan pembelian spontan di luar rencana anggaran keuangan bulanan yang sudah dicanangkan, serta memiliki frekuensi transaksi yang sangat tinggi."

        cluster_mapping[int(idx)] = {
            "label": label,
            "description": desc,
            "metrics_summary": {
                "saving_rate": round(float(row['saving_rate_raw']), 2),
                "wants_ratio": round(float(row['wants_ratio_raw']), 2),
                "unbudgeted_ratio": round(float(row['is_unbudgeted']), 2),
                "avg_transactions": round(float(row['transaction_count']), 1)
            }
        }

    monthly['persona_label'] = monthly['cluster'].map(lambda x: cluster_mapping[x]['label'])

    # =====================================================================
    # 6. PROFESSIONAL DATA VISUALIZATION
    # =====================================================================
    print("🎨 Membuat visualisasi distribusi klaster dengan PCA (2D Projection)...")
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    monthly['pca_1'] = X_pca[:, 0]
    monthly['pca_2'] = X_pca[:, 1]

    plt.figure(figsize=(10, 7))
    sns.scatterplot(
        x='pca_1', y='pca_2', hue='persona_label', data=monthly,
        palette='Set2', alpha=0.7, edgecolor='k', s=40
    )
    plt.title(f'Proyeksi Spatial Cluster Persona via PCA (k={optimal_k})')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend(title='Persona Finansial', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "cluster_pca_projection.png"), dpi=150)
    plt.close()

    print("📊 Visualisasi komparasi karakteristik metrik kunci antar-persona...")
    key_metrics = ['saving_rate_raw', 'wants_ratio_raw', 'is_unbudgeted']
    melted_df = monthly.melt(id_vars=['persona_label'], value_vars=key_metrics,
                             var_name='Metric', value_name='Value')

    plt.figure(figsize=(12, 6))
    sns.barplot(x='Metric', y='Value', hue='persona_label', data=melted_df, palette='Set2', errorbar=None)
    plt.title('Perbandingan Rata-rata Karakteristik Finansial Antar-Persona')
    plt.xlabel('Indikator Fitur')
    plt.ylabel('Nilai Rata-rata (Skala Asli)')
    plt.xticklabels = ['Saving Rate', 'Wants Ratio', 'Unbudgeted Ratio']
    plt.legend(title='Persona')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "persona_metrics_comparison.png"), dpi=150)
    plt.close()

    # =====================================================================
    # 7. EXPORTING ARTEFAKS & METADATA
    # =====================================================================
    print("💾 Menyimpan seluruh file konfigurasi dan payload JSON...")

    with open(os.path.join(OUTPUT_DIR, "cluster_profiles.json"), "w", encoding="utf-8") as f:
        json.dump(cluster_mapping, f, ensure_ascii=False, indent=4)

    with open(os.path.join(OUTPUT_DIR, "cluster_features.json"), "w", encoding="utf-8") as f:
        json.dump(features_to_cluster, f, ensure_ascii=False, indent=4)

    monthly.to_csv(os.path.join(OUTPUT_DIR, "monthly_clustered.csv"), index=True)

    print("✨ [SUCCESS] Seluruh berkas biner, konfigurasi JSON, dan gambar grafik visualisasi tersimpan di folder:", OUTPUT_DIR)
    return monthly, cluster_mapping

# Contoh cara pemanggilan pipeline penuh di Notebook Anda:
df = pd.read_excel('financial_health_combined_105k.xlsx')
final_data, persona_rules = run_complete_model_2_pipeline(df, target_k=3)

🚀 [START] Memulai Complete Pipeline Model 2 (Clustering Persona)...
📊 Mengagregasi data transaksi ke level bulanan per pengguna...
🍕 Mentransformasikan proporsi pengeluaran berdasarkan kategori...
🔍 Mengevaluasi nilai K terbaik (k=2 hingga k=6)...
💡 Menggunakan nilai K yang ditentukan manual: k=3
🤖 Menjalankan model final K-Means dengan k=3...
🏷️ Melakukan profiling centroid untuk pemetaan persona keuangan otomatis...
🎨 Membuat visualisasi distribusi klaster dengan PCA (2D Projection)...
📊 Visualisasi komparasi karakteristik metrik kunci antar-persona...
💾 Menyimpan seluruh file konfigurasi dan payload JSON...
✨ [SUCCESS] Seluruh berkas biner, konfigurasi JSON, dan gambar grafik visualisasi tersimpan di folder: artifacts
